In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

CIK = "0000789019"  # Microsoft
YEARS = {2021, 2022, 2023, 2024, 2025}

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov",
}

URL = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"

METRICS = {
    "Revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "Gross Profit": ["GrossProfit"],
    "Operating Income": ["OperatingIncomeLoss"],
    "Net Income": ["NetIncomeLoss"],
    "EPS Diluted": ["EarningsPerShareDiluted"],
    "Operating Cash Flow": ["NetCashProvidedByUsedInOperatingActivities"],
    "Capital Expenditures": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "Cash": ["CashAndCashEquivalentsAtCarryingValue"],
    "Current Assets": ["AssetsCurrent"],
    "Current Liabilities": ["LiabilitiesCurrent"],
    "Total Assets": ["Assets"],
    "Total Liabilities": ["Liabilities"],
    "Total Equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
}

def download_companyfacts():
    r = requests.get(URL, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def collect_annual_facts(us_gaap, tag_candidates, years=YEARS):
    rows = []

    for tag_rank, tag in enumerate(tag_candidates):
        fact = us_gaap.get(tag)
        if not fact:
            continue

        for unit, items in fact.get("units", {}).items():
            for item in items:
                fy = item.get("fy")
                fp = item.get("fp")
                form = item.get("form")

                if fy in years and fp == "FY" and form in {"10-K", "10-K/A"}:
                    rows.append(
                        {
                            "tag": tag,
                            "tag_rank": tag_rank,
                            "fy": fy,
                            "value": item.get("val"),
                            "unit": unit,
                            "filed": item.get("filed"),
                            "end": item.get("end"),
                        }
                    )

    if not rows:
        return pd.DataFrame(columns=["fy", "value", "tag", "unit", "filed", "end"])

    df = pd.DataFrame(rows)
    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
    df = df.sort_values(["fy", "tag_rank", "filed"], ascending=[True, True, False])

    # Keep the best row for each fiscal year
    best = df.groupby("fy", as_index=False).first()
    return best[["fy", "value", "tag", "unit", "filed", "end"]]

def main():
    data = download_companyfacts()
    us_gaap = data["facts"]["us-gaap"]

    output = {}

    for metric_name, tags in METRICS.items():
        df = collect_annual_facts(us_gaap, tags)
        output[metric_name] = df.set_index("fy")["value"].to_dict()

    result = pd.DataFrame(output).sort_index()

    # Add a simple FCF calculation if both inputs exist
    if "Operating Cash Flow" in result.columns and "Capital Expenditures" in result.columns:
        result["Free Cash Flow"] = result["Operating Cash Flow"] - result["Capital Expenditures"]

    result.index.name = "Fiscal Year"

    out_dir = Path("data/processed")
    out_dir.mkdir(parents=True, exist_ok=True)

    result.to_csv(out_dir / "msft_financials_5y.csv")
    print(result)

if __name__ == "__main__":
    main()

                  Revenue  Gross Profit  Operating Income   Net Income  \
Fiscal Year                                                              
2021         125843000000   82933000000       42959000000  39240000000   
2022         143015000000   96937000000       52959000000  44281000000   
2023         168088000000  115856000000       69916000000  61271000000   
2024         198270000000  135620000000       83383000000  72738000000   
2025         211915000000  146052000000       88523000000  72361000000   

             EPS Diluted  Operating Cash Flow  Capital Expenditures  \
Fiscal Year                                                           
2021                5.06          52185000000           13925000000   
2022                5.76          60675000000           15441000000   
2023                8.05          76740000000           20622000000   
2024                9.65          89035000000           23886000000   
2025                9.68          87582000000          

In [4]:
import pandas as pd

df = pd.read_csv("msft_financials_5y.csv")
df.head()

,Fiscal Year,Revenue,Gross Profit,Operating Income,Net Income,EPS Diluted,Operating Cash Flow,Capital Expenditures,Cash,Current Assets,Current Liabilities,Total Assets,Total Liabilities,Total Equity,Free Cash Flow
0,2021,125843000000,82933000000,42959000000,39240000000,5.06,52185000000,13925000000,13576000000,181915000000,72310000000,301311000000,183007000000,102330000000,38260000000
1,2022,143015000000,96937000000,52959000000,44281000000,5.76,60675000000,15441000000,14224000000,184406000000,88657000000,333779000000,191791000000,118304000000,45234000000
2,2023,168088000000,115856000000,69916000000,61271000000,8.05,76740000000,20622000000,13931000000,169684000000,95082000000,364840000000,198298000000,141988000000,56118000000
3,2024,198270000000,135620000000,83383000000,72738000000,9.65,89035000000,23886000000,34704000000,184257000000,104149000000,411976000000,205753000000,166542000000,65149000000
4,2025,211915000000,146052000000,88523000000,72361000000,9.68,87582000000,28107000000,18315000000,159734000000,125286000000,512163000000,243686000000,206223000000,59475000000
